In [1]:
import tensorflow as tf
import tensorflow.keras as keras
import pickle
import functools

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.8
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from network import build_vit
from train import WarmUpCosine, AdamW, LastNSaver, cifar_pad_crop_flip, mixup_batch, randaugment_mild, make_test_ds
from noise import noise_a, noise_p

In [4]:
initial_lr = 2e-4
weight_decay = 1e-5
epochs = 200
warmup_epochs = 5
batch_size = 128
image_size = 32

In [5]:
with open('data.pkl', 'rb') as f:
    [x_train,y_train,x_test,y_test]=pickle.load(f)
y_train_onehot=tf.keras.utils.to_categorical(y_train,num_classes=4)
y_test_onehot=tf.keras.utils.to_categorical(y_test,num_classes=4)

In [6]:
model = build_vit(num_classes=4)

In [7]:
total_steps = epochs * (x_train.shape[0] // batch_size)
warmup_steps = warmup_epochs * (x_train.shape[0] // batch_size)
lr_schedule = WarmUpCosine(initial_lr, total_steps, warmup_steps)
optimizer = AdamW(
    learning_rate=lr_schedule,
    weight_decay=weight_decay
)

In [8]:
loss_fn = tf.keras.losses.CategoricalCrossentropy()
model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=['accuracy']
)

In [9]:
saver = LastNSaver(n=20)

In [14]:
gaussian_noise_batch = functools.partial(noise_a, a=0.2)
salt_pepper_noise_batch = functools.partial(noise_p, n=64)

In [15]:
def random_noise_batch(x, p_gaussian=0.5):
    x_g = gaussian_noise_batch(x)
    x_sp = salt_pepper_noise_batch(x)

    b = tf.shape(x)[0]
    use_g = tf.random.uniform([b, 1, 1, 1]) < p_gaussian
    use_g = tf.cast(use_g, x.dtype)

    return use_g * x_g + (1.0 - use_g) * x_sp


def clean_noisy_batch(x, y):
    x_noisy = random_noise_batch(x)

    x = tf.concat([x, x_noisy], axis=0)
    y = tf.concat([y, y], axis=0)
    return x, y

In [16]:
def make_train_ds(x_train, y_train_onehot, batch_size=128,
                  image_size=32, pad=4,
                  mixup_alpha=0.0,
                  strong_aug=False,
                  use_noise=True):

    ds = tf.data.Dataset.from_tensor_slices((x_train, y_train_onehot))
    ds = ds.shuffle(min(len(x_train), 20000), reshuffle_each_iteration=True)

    def to_float(x, y):
        x = tf.cast(x, tf.float32)
        y = tf.cast(y, tf.float32)
        return x, y

    ds = ds.map(to_float, num_parallel_calls=AUTOTUNE)

    ds = ds.map(
        lambda x, y: cifar_pad_crop_flip(x, y, image_size=image_size, pad=pad),
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.map(
        lambda x, y: randaugment_mild(x, y, strong_aug=strong_aug),
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(batch_size, drop_remainder=True)

    if use_noise:
        ds = ds.map(clean_noisy_batch, num_parallel_calls=AUTOTUNE)

    if mixup_alpha and mixup_alpha > 0:
        ds = ds.map(
            lambda xb, yb: mixup_batch(xb, yb, alpha=mixup_alpha),
            num_parallel_calls=AUTOTUNE
        )

    ds = ds.prefetch(AUTOTUNE)
    return ds

In [17]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = make_train_ds(
    x_train, y_train_onehot,
    batch_size=batch_size,
    image_size=32,
    pad=4,
    mixup_alpha=0.2,
    strong_aug=False,
    use_noise=True
)
test_ds = make_test_ds(x_test, y_test_onehot, batch_size=batch_size)

In [18]:
model.fit(
    train_ds,
    epochs=epochs,
    validation_data=test_ds,
    verbose=2,
    callbacks=[saver]
)

Epoch 1/200
156/156 - 24s - loss: 1.3180 - accuracy: 0.3727 - val_loss: 1.8390 - val_accuracy: 0.3058 - 24s/epoch - 152ms/step
Epoch 2/200
156/156 - 14s - loss: 1.1876 - accuracy: 0.4756 - val_loss: 1.6913 - val_accuracy: 0.2785 - 14s/epoch - 88ms/step
Epoch 3/200
156/156 - 14s - loss: 1.1306 - accuracy: 0.5193 - val_loss: 1.0652 - val_accuracy: 0.5255 - 14s/epoch - 88ms/step
Epoch 4/200
156/156 - 14s - loss: 1.0901 - accuracy: 0.5506 - val_loss: 0.9661 - val_accuracy: 0.5885 - 14s/epoch - 88ms/step
Epoch 5/200
156/156 - 14s - loss: 1.0611 - accuracy: 0.5731 - val_loss: 0.8983 - val_accuracy: 0.6283 - 14s/epoch - 88ms/step
Epoch 6/200
156/156 - 14s - loss: 1.0225 - accuracy: 0.5900 - val_loss: 0.8572 - val_accuracy: 0.6518 - 14s/epoch - 88ms/step
Epoch 7/200
156/156 - 14s - loss: 0.9979 - accuracy: 0.6086 - val_loss: 0.8096 - val_accuracy: 0.6653 - 14s/epoch - 88ms/step
Epoch 8/200
156/156 - 14s - loss: 0.9541 - accuracy: 0.6307 - val_loss: 0.8187 - val_accuracy: 0.6705 - 14s/epoch - 8

In [19]:
model.save("ViT_8.h5")